In [8]:
import numpy as np
import matplotlib.pyplot as plt
import re

import ase.io as aio


In [2]:
def get_initial_info(trj):
    pattern= r"TIMESTEP"
    header=8
    N_H=0
    N_O=0
    N_Fe=0
    #SCV structure (1uc)
    N_OS_uc=8    # number oxygen in first outer layer/uc
    N_FeS_uc=4   # number of iron in first outer layer/uc
    N_FeS2_uc=3  # number of iron  second/outer layer/uc
    with open(trj, 'r') as file:

        for count, line in enumerate(file):     
            if  re.search(pattern, line) and count !=0 :
                break
                
            split_line=line.split()   
            if count == 3:
                N_atom=float(split_line[0])
            if count == 5:
                L_x=float(split_line[1])
            if count == 7:
                L_z=float(split_line[1])
            #position block
            if count > header:        
                if split_line[1] == '1':
                   N_H+=1
                if split_line[1] == '2':
                   N_O+=1   
                if split_line[1] == '3':
                   N_Fe+=1   
        L_y=L_x
        
        #replica
        for n in range(1,10):
            if 8 < L_x/n < 9:
                n_replica=n
                
        if N_atom != N_H+N_O+N_Fe:
            print("ATOM LOST")
        
        f=n_replica**2
        N_water=int(N_H/2)
        N_Fe_uc=int(N_Fe/f)
        N_O_uc=int((N_O-N_water)/f)
        w_uc=float(N_water/f/2)
        N_OS=N_OS_uc*f
        N_FeS=N_FeS_uc*f
        N_FeS2= N_FeS2_uc*f
        diag_lenght=np.sqrt(L_x**2+L_y**2)
        print("L_x = L_y = ", L_x)
        print("L_z = ", L_z)
        print("Diagonal= ", diag_lenght)
        print("N_O_uc = ",  N_O_uc)
        print("N_Fe_uc =",  N_Fe_uc)
        print("w_uc =",  w_uc)
        print(f"Size factor f = {f}")
        print(f"N_water = {N_water}")
        print(f"N_H = {N_H}")
        print(f"N_O = {N_O}")
        print(f"N_Fe = {N_Fe}")
        print(f"N_atom = {N_atom}")
        print(f"N_O surface = {N_OS}")
        print(f"N_Fe surface = {N_FeS}")
        print(f"N_Fe surface2 = {N_FeS2}")
        
        return(f, L_x, L_y, L_z, diag_lenght, N_water,  N_H, N_O, N_Fe, N_OS, N_FeS, N_FeS2)

In [3]:
def get_atom_numers(w_uc,  N_O_mag_uc, N_Fe_mag_uc, N_OS_uc, N_FeS_uc, N_FeS2_uc ):
    #calculate number of atom in the whole config, of every species and per surface.
    #determine the size factor "f": f_small=3x3=9; f_big=6x6=36;   f_full_coverage=4x4=16: 
    
    print(f"Water/uc/Surface = {w_uc}")
    if w_uc in (1,2,3,4):
        f=9
    elif w_uc in (5,6,7,8):
        f=36
    elif w_uc==68:
        f=16
    print(f"size_factor = {f}")
    
    
    if w_uc in (1,2,3,4):
        L_x=25.08508
        L_y=25.08508
        L_z=28.85502 
    elif w_uc in (5,6,7,8):
        L_x =50.17016 
        L_y =50.17016 
        L_z =34.62602 
    elif w_uc==68:
        L_x=33.584 
        L_y=33.584 
        L_z=71.47838

    diag_lenght=np.sqrt(L_x**2+L_y**2) #simulation box xy diagonal lenght to add boundaries 
    print("diag_lenght=" , diag_lenght)
    
    
    
    
    
    
    #species
    N_water=w_uc*f*2         #*2 surfaces
    N_H=N_water*2             
    N_O_mag=N_O_mag_uc*f
    N_O=N_O_mag+N_water
    N_Fe=N_Fe_mag_uc*f
    N_atom=N_H+N_O+N_Fe

    #surface
    N_OS=N_OS_uc*f
    N_FeS=N_FeS_uc*f
    N_FeS2= N_FeS2_uc*f

    

    print(f"N_water = {N_water}")
    print(f"N_H = {N_H}")
    print(f"N_O = {N_O}")
    print(f"N_Fe = {N_Fe}")
    print(f"N_atom = {N_atom}")
    print(f"N_O surface = {N_OS}")
    print(f"N_Fe surface = {N_FeS}")
    print(f"N_Fe surface2 = {N_FeS2}")
    return(f, diag_lenght, L_x, L_y, L_z,  N_water, N_H, N_O, N_Fe, N_OS, N_FeS, N_FeS2 )

In [4]:
def parser(t_slice,  N_water, N_O, N_OS, N_Fe, N_FeS, N_FeS2):
    
    H_u=[]
    H_d=[]
    O_u=[]
    O_d=[]
    OS_u=[]
    OS_d=[]
    FeS_u=[]
    FeS_d=[]
    FeS2_u=[]
    FeS2_d=[]
    
    
    positions=t_slice.get_positions(wrap=True)
   
    #indices species 
    H_indices = np.where(t_slice.get_atomic_numbers()==1)[0]
    O_indices = np.where(t_slice.get_atomic_numbers()==2 )[0]
    Fe_indices = np.where(t_slice.get_atomic_numbers()==3 )[0]
 
    
    #H
    H_u=positions[H_indices[:N_water]]
    H_d=positions[H_indices[N_water:]]
    #O 
    #water_u
    O_u= positions[O_indices[:int(N_water/2)]] 
    OS_u= positions[O_indices[int(N_water/2):int(N_water/2)+N_OS]] 
    OS_d=positions[O_indices[ N_O -int(N_water/2) -N_OS:  N_O -int(N_water/2)   ]] 
    O_d=positions[O_indices[  N_O -int(N_water/2) :  ]] 
   
    #Fe
    FeS_u= positions[Fe_indices[:N_FeS]]    #surface up
    FeS2_u= positions[Fe_indices[N_FeS: N_FeS+ N_FeS2 ]]     #2 surface up
    FeS2_d= positions[Fe_indices[ N_Fe - N_FeS- N_FeS2 : N_Fe - N_FeS ] ]   #2 surface down
    FeS_d= positions[Fe_indices[N_Fe - N_FeS :]]
  
    
    #checks
    if len(H_u) != N_water:
      print(f"H Err:  {len(H_u) - N_water} atom" )
    if len(H_d) != N_water:
      print(f"H Err:  {len(H_d) - N_water} atom" )
    if len(O_u) != N_water/2:
      print(f"O Err: {len(O_u) - N_water/2} atom" ) 
    if len(O_d) != N_water/2:
      print(f"O Err: {len(O_d) - N_water/2} atom" ) 
    if len(OS_u) != N_OS:
      print(f"OS_u Err: {len(OS_u) - N_OS} atom" )
    if len(OS_d) != N_OS:
      print(f"OS_d Err: {len(OS_d) - N_OS} atom" )
    if len(FeS_u) != N_FeS:
      print(f"FeS_u Err: {len(FeS_u) - N_FeS} atom" )
    if len(FeS_d) != N_FeS:
      print(f"FeS_d Err: {len(FeS_d) - N_FeS} atom" )
    if len(FeS2_u) != N_FeS2:
      print(f"FeS2_u Err: {len(FeS2_u) - N_FeS2} atom" )
    if len(FeS2_d) != N_FeS2:
      print(f"FeS2_d Err: {len(FeS2_d) - N_FeS2} atom" )
     
    
   
    return H_u, H_d, O_u, O_d, OS_u, OS_d, FeS_u, FeS_d, FeS2_u, FeS2_d

In [5]:
#give #H or #O of water, z of the surface up and down , and cut distance
# 2) (surface u) shifting z to zero, (surface d) shifting  z to zero  and inverting
def surface_up_down_translate(A_u, A_d, z_u, z_d,  max_distance):
    
    tmp_u=[]
    tmp_d=[]
  
    for atom in A_u:
        #if  z_u  <atom[2]<  z_u + max_distance  :
            tmp_u.append( [atom[0], atom[1],    atom[2]-z_u ])
            
    for atom in A_d:    
       #if z_d -  max_distance < atom[2]<  z_d:
            tmp_d.append([atom[0], atom[1], -atom[2]+z_d])
  


    return np.array(tmp_u), np.array(tmp_d)

def up_down_z_all_t_extractor(H_u, H_d ):
    zH_ud=[]
    for atom in H_u:
            zH_ud.append(atom[2])
    for atom in H_d:
            zH_ud.append(atom[2])

   # print(len(zH_ud))
    return zH_ud

#determine exactly the surfaces positions with average of z of all atom in the surface: give atoms in a layer
def get_layer_z(A):
    z=A[:,2]
    return np.mean(z), max(z), min(z)

In [6]:
#oct and tet analysis based on oxygens surface layer: OS_u or OS_d as input, box xy diagonal, and  cluster parm epsilon =1

#output diagonal O raws displacment to separate (Fe) oct and tet zones
def get_diagonal_O_raws(OS_u, control, diag_lenght,  epsilon):
    diagonal_projection=[]

    
    for atom in OS_u:
        r=np.sqrt(atom[0]**2+atom[1]**2)
        theta=np.arctan(atom[1]/atom[0])
        #print(theta)
        if control == 'u':
            diagonal_projection.append(r*np.cos(theta - (np.pi/4)  )   )
        elif control == 'd':
            diagonal_projection.append(r*np.sin(theta - (np.pi/4)  )   )
        else:
            print("control must be 'u' or 'd'")
            return

    #clusterization of diagonal projection of O position in raws
    displacement=[]
    
    displacement.append([diagonal_projection[0]])

    for i, proj in enumerate(diagonal_projection[1:]):
        copy=displacement
     
        for j in range(len(copy)):
            #print(j)
            if np.abs(proj-displacement[j][0])<epsilon:
                displacement[j].append(proj)
                #print("existing")
                break
            elif j==len(copy)-1:      #last round and no match found create new entry
                #print("new")
                displacement.append([proj])


    #average over the O diagonal raws
    mean_displacement=[]   
    for i in displacement:
        mean_displacement.append(np.mean(i))
    mean_displacement=list(np.sort(mean_displacement))
   
    if control == 'u':
        mean_displacement.insert(0,0)
        mean_displacement.append(diag_lenght)
    else:
        mean_displacement.insert(0,-diag_lenght/2)
        mean_displacement.append(diag_lenght/2)
    #print(mean_displacement)
    return list(mean_displacement)


#oct/tet regions atomic separation; inside O raws = oct, outside = tet: 
def tet_oct_split(O_u, diag_raw , control ):
      
    O_oct_u=[]
    O_tet_u=[]
   
    for atom in O_u:
        r=np.sqrt(atom[0]**2+atom[1]**2)
        theta=np.arctan(atom[1]/atom[0])
    
        if control == 'u':
            diagonal_projection=(r*np.cos(theta - (np.pi/4)  ))     #up cos raws   \\\\\           
        elif control == 'd':
            diagonal_projection=(r*np.sin(theta - (np.pi/4)  ))     #down sin raws  /////   
            
        else:  
            print("control must be 'u' or 'd'")
            return   
        
        if diagonal_projection <diag_raw[0] or diagonal_projection > diag_raw[-1]:  #inbound check
                print("OUT of bounds: diag =" , diagonal_projection, "cartesian =", atom)
                
        if (diag_raw[2]-diag_raw[1])<(diag_raw[3]-diag_raw[2]):           #check if the first raw (the zero is the border)is oct or tet (if T == oct if F ==tet )
            # first raw is oct
            #if diagonal projection in (diag_raw[2n],diag_raw[2n+1])   then tet region  even!
            #if diagonal projection in (diag_raw[2n+1],diag_raw[2n+2]) then oct region odd!
            for n in range(len(diag_raw)-1):  
                if diag_raw[n] <= diagonal_projection <= diag_raw[n+1]: 
                    if n % 2 ==0:
                        O_tet_u.append(atom) #n even ==> outside O raws == tet region
                    else:
                        O_oct_u.append(atom) #n odd ==> inside O raws == oct region
                    break
        else:
            #first raw is tet
            for n in range(len(diag_raw)-1):  
                if diag_raw[n] <= diagonal_projection <= diag_raw[n+1]: 
                    if n % 2 ==0:
                        O_oct_u.append(atom) #n even ==> inside O raws == oct region
                    else:
                        O_tet_u.append(atom) #n odd ==> outside O raws == tet region
                    break
        
    if len(O_oct_u)+len(O_tet_u) != len(O_u):
        print("MISSING ATOM: ",len(O_u)-  (len(O_oct_u)+len(O_tet_u))) 
    return O_oct_u,  O_tet_u



#oct/tet regions atomic separation; inside O raws = oct, outside = tet: 
def oct_or_tet(atom, diag_raw, control):
    
    
    r=np.sqrt(atom[0]**2+atom[1]**2)
    theta=np.arctan(atom[1]/atom[0])

    if control == 'u':
        diagonal_projection=(r*np.cos(theta - (np.pi/4)  ))     #up cos raws   \\\\\           
    elif control == 'd':
        diagonal_projection=(r*np.sin(theta - (np.pi/4)  ))     #down sin raws  /////   
        
    else:  
        print("control must be 'u' or 'd'")
        return   
    
    if diagonal_projection <diag_raw[0] or diagonal_projection > diag_raw[-1]:  #inbound check
            print("OUT of bounds: diag =" , diagonal_projection, "cartesian =", atom)
            
    if (diag_raw[2]-diag_raw[1])<(diag_raw[3]-diag_raw[2]):           #check if the first raw (the zero is the border)is oct or tet (if T == oct if F ==tet )
        # first raw is oct
        #if diagonal projection in (diag_raw[2n],diag_raw[2n+1])   then tet region  even!
        #if diagonal projection in (diag_raw[2n+1],diag_raw[2n+2]) then oct region odd!
        for n in range(len(diag_raw)-1):  
            if diag_raw[n] <= diagonal_projection <= diag_raw[n+1]: 
                if n % 2 == 0:
                    oct_bool=False  #n even ==> outside O raws == tet region
                else:
                    oct_bool=True #n odd ==> inside O raws == oct region
                break
    else:
        #first raw is tet
        for n in range(len(diag_raw)-1):  
            if diag_raw[n] <= diagonal_projection <= diag_raw[n+1]: 
                if n % 2 ==0:
                    oct_bool=True #n even ==> inside O raws == oct region
                else:
                     oct_bool=False  #n odd ==> outside O raws == tet region
                break
     
    return oct_bool

In [22]:
def get_distinguished_cluster(clustering):
        distinguished_clusters=[]
        cluster_labels=[]
        #for cluster_index in range(-1,N_cluster):
        for O_index, cluster_index in enumerate(clustering): 
             if cluster_index==-1:
                  distinguished_clusters.append([O_index])
             else:     
                  if cluster_index in cluster_labels:
                       insertion_index=cluster_labels.index(cluster_index)
                       distinguished_clusters[insertion_index].append(O_index)
                  else:
                       cluster_labels.append(cluster_index)
                       insertion_index=cluster_labels.index(cluster_index)
                       distinguished_clusters.insert(insertion_index, [O_index])
        #distinguished_clusters=sorted(distinguished_clusters, key=len)
        return distinguished_clusters 



def distance_matrix(O, O_indx):
     #Creating "Sparse" (0 and 1) Distance matrix
     dist_matrix = np.zeros((O.shape[0],O.shape[0]))
     for i, indices in enumerate(O_indx):
         for j in indices:
             if j != i:
                     dist_matrix[i,j] = 1
                     #dist_matrix[j,i] = 1
     dist_matrix_sparse = csr_matrix(dist_matrix)
     
     return dist_matrix_sparse
            

In [35]:
import h5py
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from sklearn.neighbors import sort_graph_by_row_values
from sklearn.cluster import DBSCAN
import warnings






def main_cluster_analysis(h5py_files_folder,L_x, L_y, L_z , control, diag_lenght, N_water , O_cut_off, surface_selection):

    p_cluster_size=[0]*int(N_water/2)
    n_len_cluster=0
    trimer_in_line=0

    trimer_line_count=0



    #H tree box for bonding
    box_size = np.array([L_x, L_y, L_z])  


    #time
    t=0

    h5py_str='.h5py'

    with h5py.File(h5py_files_folder+"H"+surface_selection+h5py_str, 'r') as f_H, h5py.File(h5py_files_folder+"O"+surface_selection+h5py_str, 'r') as f_O, h5py.File(h5py_files_folder+"OS"+surface_selection+h5py_str, 'r') as f_OS:
        for k in f_H.keys(): #main time loop
        
            H = f_H[k][:]
            O = f_O[k][:]
            OS = f_OS[k][:]

            if t %10 == 0:   #it does every 10 steps
                #get O raws info
                diag_raw=get_diagonal_O_raws(OS, control, diag_lenght, 1 ) #give OS or OS_d, control u or d accordingly, diag_lenght and cluster param
         

            #Oxygen treee
            tree = cKDTree(data=O, leafsize=O.shape[0], boxsize= box_size)  #makes a tree of Hs
            O_indx = tree.query_ball_point(O, O_cut_off) 
            #Dist matrix
            dist_matrix_sparse=  distance_matrix(O, O_indx)
            
            #Clustering
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                dist_matrix_sparse_sorted = sort_graph_by_row_values(dist_matrix_sparse)
                clustering = DBSCAN(eps= 1.1, min_samples=2, metric="precomputed").fit_predict(dist_matrix_sparse_sorted)

            #Analysis
            #cluster i=-1: alone cluster
          
            distinguished_clusters =get_distinguished_cluster(clustering)
           
            n_len_cluster+=len(distinguished_clusters)
            
            
            for cl in distinguished_clusters:
                if len(cl)==3:
                        trimer_line_count+=1
                        oct_bool_check_all=True
                        for i in cl:
                             #print(O[i])
                             oct_bool=oct_or_tet(O[i],diag_raw,control)
                             oct_bool_check_all=oct_bool_check_all*oct_bool
                        if  oct_bool_check_all:
                             trimer_in_line+=1


                for i in range(len(p_cluster_size)):
                        if len(cl)==i+1:
                            p_cluster_size[i]+=1

                            break
                        
                        
            t+=1
        

    return p_cluster_size, trimer_in_line/trimer_line_count, n_len_cluster
     

In [65]:
#SINGLE RUN
#IMPORTANT: trj must be ordered by z and clustered by atom

path_of_run='/data/home/salvatorer26/projects/ferrite_oxide/production/production_runs/water_3_uc/run_4/'
file_trj=path_of_run+"trjmagnetite.dump"
h5py_files_folder= path_of_run+"trajectory_h5py_files/" 



f, L_x, L_y, L_z, diag_lenght, N_water, N_H, N_O, N_Fe, N_OS, N_FeS, N_FeS2= get_initial_info(file_trj)

O_cut_off = 3.36

max_distance=5


control='u'
surface_selection='_u'


prob_cluster_size, prob_linetrimer, n_len_cluster = main_cluster_analysis(h5py_files_folder,L_x, L_y, L_z , control, diag_lenght, N_water , O_cut_off, surface_selection)





L_x = L_y =  25.08508
L_z =  28.85502
Diagonal=  35.47566034921408
N_O_uc =  64
N_Fe_uc = 44
w_uc = 3.0
Size factor f = 9
N_water = 54
N_H = 108
N_O = 630
N_Fe = 396
N_atom = 1134.0
N_O surface = 72
N_Fe surface = 36
N_Fe surface2 = 27


In [67]:


for i, p in enumerate(prob_cluster_size):
    p_norm=p/n_len_cluster
    print( f"P(Cl_{i+1})="  ,"{:.2f}".format(p_norm))
    
print("P(line_trimer)=" , "{:.2f}".format(prob_linetrimer))




P(Cl_1)= 0.03
P(Cl_2)= 0.50
P(Cl_3)= 0.06
P(Cl_4)= 0.04
P(Cl_5)= 0.02
P(Cl_6)= 0.01
P(Cl_7)= 0.02
P(Cl_8)= 0.01
P(Cl_9)= 0.01
P(Cl_10)= 0.01
P(Cl_11)= 0.01
P(Cl_12)= 0.01
P(Cl_13)= 0.01
P(Cl_14)= 0.01
P(Cl_15)= 0.01
P(Cl_16)= 0.01
P(Cl_17)= 0.01
P(Cl_18)= 0.01
P(Cl_19)= 0.01
P(Cl_20)= 0.02
P(Cl_21)= 0.03
P(Cl_22)= 0.02
P(Cl_23)= 0.08
P(Cl_24)= 0.00
P(Cl_25)= 0.05
P(Cl_26)= 0.00
P(Cl_27)= 0.00
P(line_trimer)= 0.85


In [17]:
distinguished_clusters

[[0, 5, 6, 8], [1, 4, 7], [2, 3]]

In [57]:
import h5py
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from sklearn.neighbors import sort_graph_by_row_values
from sklearn.cluster import DBSCAN
import warnings

control='d'


max_distance=5 #cut distance from each surface: <26 


p_cluster_size=[0]*int(N_water/2)
n_len_cluster=0
trimer_in_line=0
          
trimer_line_count=0



#H tree box for bonding
box_size = np.array([L_x, L_y, L_z])  
O_cut_off = 3.36


#time
t=0

#up
surface_selection='_d'
h5py_str='.h5py'

with h5py.File(h5py_files_folder+"H"+surface_selection+h5py_str, 'r') as f_H, h5py.File(h5py_files_folder+"O"+surface_selection+h5py_str, 'r') as f_O, h5py.File(h5py_files_folder+"OS"+surface_selection+h5py_str, 'r') as f_OS:
    for k in f_H.keys(): #main time loop
      
          H = f_H[k][:]
          O = f_O[k][:]
          OS = f_OS[k][:]
     
          if t %10 == 0:   #it does every 10 steps
               #get O raws info
               diag_raw=get_diagonal_O_raws(OS, control, diag_lenght, 1 ) #give OS or OS_d, control u or d accordingly, diag_lenght and cluster param
          
          #split oct/tet
          O_oct, O_tet=tet_oct_split(O, diag_raw , control )     
          H_oct, H_tet=tet_oct_split(H, diag_raw , control )
             
       
          #get O raws info (sporadic)
          diag_raw=get_diagonal_O_raws(OS, control, diag_lenght, 1 ) #give OS or OS_d, control u or d accordingly, diag_lenght and cluster param
       

          tree = cKDTree(data=O, leafsize=O.shape[0], boxsize= box_size)  #makes a tree of Hs
          O_indx = tree.query_ball_point(O, O_cut_off) 

          #tree = cKDTree(data=O_d, leafsize=O_d.shape[0], boxsize= box_size)  #makes a tree of Hs
          #O_d_indx = tree.query_ball_point(O_d, O_cut_off) 

          dist_matrix = np.zeros((O.shape[0],O.shape[0]))
          for i, indices in enumerate(O_indx):
               for j in indices:
                    if j != i:
                         dist_matrix[i,j] = 1
                         #dist_matrix[j,i] = 1

          dist_matrix_sparse = csr_matrix(dist_matrix)
          with warnings.catch_warnings():
               warnings.simplefilter("ignore")
               dist_matrix_sparse_sorted = sort_graph_by_row_values(dist_matrix_sparse)
               clustering = DBSCAN(eps= 1.1, min_samples=2, metric="precomputed").fit_predict(dist_matrix_sparse_sorted)
          
          #cluster i=-1: alone cluster
          N_cluster=int(np.max(clustering))+1
          
          distinguished_clusters=[]
          cluster_labels=[]
          #for cluster_index in range(-1,N_cluster):
          for O_index, cluster_index in enumerate(clustering): 
               if cluster_index==-1:
                    distinguished_clusters.append([O_index])
               else:     
                    if cluster_index in cluster_labels:
                         insertion_index=cluster_labels.index(cluster_index)
                         distinguished_clusters[insertion_index].append(O_index)
                    else:
                         cluster_labels.append(cluster_index)
                         insertion_index=cluster_labels.index(cluster_index)
                         distinguished_clusters.insert(insertion_index, [O_index])
          
          
          #distinguished_clusters=sorted(distinguished_clusters, key=len)
          n_len_cluster+=len(distinguished_clusters)
          for cl in distinguished_clusters:
               
               if len(cl)==3:
                    trimer_line_count+=1
                    oct_bool_check_all=True
                    for i in cl:
                         #print(O[i])
                         oct_bool=oct_or_tet(O[i],diag_raw,control)
                         oct_bool_check_all=oct_bool_check_all*oct_bool
                    if  oct_bool_check_all:
                         trimer_in_line+=1
                        
                        
               for i in range(len(p_cluster_size)):
                    if len(cl)==i+1:
                        p_cluster_size[i]+=1
                        
                        break
      
              
          t+=1
        

     

In [58]:


for i, p in enumerate(prob_cluster_size):
    p_norm=p/n_len_cluster
    print( f"P(Cl_{i})="  ,"{:.2f}".format(p_norm))
    
print("P(line_trimer)=" , "{:.2f}".format(prob_linetrimer))

P(Cl_0)= 0.06
P(Cl_1)= 0.40
P(Cl_2)= 0.07
P(Cl_3)= 0.05
P(Cl_4)= 0.05
P(Cl_5)= 0.03
P(Cl_6)= 0.23
P(Cl_7)= 0.00
P(Cl_8)= 0.00
P(line_trimer)= 0.05


In [55]:
trimer_in_line/trimer_line_count

uno=0
for p in p_cluster_size:
    p_norm=p/n_len_cluster
    print(p_norm)
    uno+=p_norm


0.08937377872894761
0.15070252163394435
0.43348841537173166
0.11625569926491114
0.10711826556248255
0.10287522099190471
0.00018609844607797525
0.0
0.0
0.9999999999999999


In [14]:
 for i in range(len(p_cluster_size)):
     print(i)

0
1
2
3
4
5
6
7
8


In [15]:
uno=0
for p in p_cluster_size:
    p_norm=p/n_len_cluster
    print(p_norm)
    uno+=p_norm
print(uno)

0.08937377872894761
0.15070252163394435
0.43348841537173166
0.11625569926491114
0.10711826556248255
0.10287522099190471
0.00018609844607797525
0.0
0.0
0.9999999999999999


In [16]:
0.08937377872894761
0.15070252163394435
0.43348841537173166
0.11625569926491114
0.10711826556248255
0.10287522099190471
0.00018609844607797525
0.0
0.0
0.9999999999999999

0.9999999999999999

In [52]:
0.06969148145433988
0.44757697260288315
0.07805613425320086
0.05554799468179772
0.0516954386993436
0.03745773180766533
0.25983815077312844
0.0
0.00013609572764104229
1.0

0.00013609572764106126